# GEMS G14.1: critical failure analysis, discovery calibration and corrected training

**Status: UNTESTED.** This notebook and the `gems_bench.g14_1` package were written without executing any code. The first run is the first test.

## 00. Research context

G14 `T1_local_cnn__da3f44d05d84` (5.2 M parameters): inner surrogate ~0.85, geographic cross-fitted DTI 0.081 (worst zone 0.075), crest_w1 at 8 %, **official leaderboard 0.0189**; the submission was calibrated on the *geographic* scorecard because no discovery scorecard existed.

Static audit (`docs/g14_1/STATIC_CODE_AUDIT.md`) found, before any run:

* **S01** every G14 training positive is also an input pixel of its known-fault context (shortcut);
* **S05** geo-only units trained, discovery task requested, submission calibrated on dense geographic truth;
* **S03 / S04** inner validation shares geography, context and target raster with training, and is a mean of per-batch ratios on positive-centred patches;
* **S06 / S07 / S08** crest_w1 area is under-reported (dilation), the export is binary, crests are found before known-fault masking.

These are hypotheses with a confirmed code path, not measured causes. This notebook measures them, from existing artifacts first (FAST: no GPU, no training), then with inference on the existing checkpoints, then with corrected training.

Execution modes (cell 01): `AUDIT_ONLY` / `DIAGNOSE` (existing artifacts, CPU), `COUNTERFACTUAL` (+ inference with existing checkpoints), `TRAIN` (+ corrected training), `SUBMIT` (write candidate files from existing maps; never uploads). Profiles: `FAST`, `STANDARD`, `FULL`.

Outputs: `/content/gems_work/g14_1/...` mirrored to `MyDrive/GEMS_2/outputs/G14_1_CRITICAL_RECOVERY/`. Offline numbers are proxies on held-out units, never leaderboard scores.

## 01. Configuration
Edit only this cell. `G14_OVERRIDES` must repeat the *semantic* OVERRIDES of the original G14 run (data / model / loss / train / split / eval sections) so the experiment hash `da3f44d05d84` is reproduced; paths and flags do not matter.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'g14-1-critical-recovery',
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'source_experiment': 'T1_local_cnn__da3f44d05d84',
    'profile': 'FAST',            # FAST | STANDARD | FULL
    'mode': 'AUDIT_ONLY',         # AUDIT_ONLY | DIAGNOSE | COUNTERFACTUAL | TRAIN | SUBMIT
    'reuse_artifacts': True,
    'resume': True,
    'force_rebuild': False,
    'stages': {
        'artifact_inventory': True, 'geometry_audit': True, 'prediction_diagnostics': True,
        'representation_sweep': True, 'champion_comparison': True, 'failure_atlas': True,
        'checkpoint_counterfactuals': False, 'train_models': False, 'transformers': False, 'submission': False,
    },
    'validation': {'primary_task': 'disc_official', 'secondary_task': 'geo', 'protocol': 'geo_and_disc',
                   'locked_test': True},
    'resources': {'device': 'auto', 'num_workers': 2, 'mixed_precision': True, 'cache_enabled': True},
    'seed': 2026,
}
G14_OVERRIDES = {}   # semantic overrides of the ORIGINAL G14 run, e.g. {'data': {...}} ; {} if defaults were used

# Named modes set compatible capabilities automatically. Examples:
#   CONFIG['mode'] = 'COUNTERFACTUAL'; CONFIG['stages']['checkpoint_counterfactuals'] = True   (inference, no training)
#   CONFIG['mode'] = 'TRAIN'; CONFIG['profile'] = 'FULL'; CONFIG['stages']['train_models'] = True
#   CONFIG['mode'] = 'SUBMIT'; CONFIG['stages']['submission'] = True   (reuses maps; writes files; never uploads)

## 02. Environment setup

In [ ]:
import importlib, subprocess, sys, time, json
need = [p for p in ('rasterio', 'skimage', 'lightgbm', 'xgboost', 'pyarrow') if importlib.util.find_spec(p) is None]
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + [{'skimage': 'scikit-image'}.get(p, p) for p in need], check=True)
try:
    import torch
    print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except ImportError:
    print('torch not installed: only CPU stages (FAST) can run')

## 03. Repository checkout

In [ ]:
import os
CODE = '/content/GEMS'
if not os.path.isdir(CODE):
    subprocess.run(['git', 'clone', '-q', '-b', CONFIG['ref'], f"https://github.com/{CONFIG['repo']}.git", CODE], check=True)
else:
    subprocess.run(['git', '-C', CODE, 'fetch', '-q', 'origin', CONFIG['ref']], check=True)
    subprocess.run(['git', '-C', CODE, 'checkout', '-q', CONFIG['ref']], check=True)
    subprocess.run(['git', '-C', CODE, 'pull', '-q', '--ff-only'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
print(subprocess.run(['git', '-C', CODE, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
# Optional, recommended once: the G14.1 tests (written but never executed by the author)
# !cd /content/GEMS && python -m pytest -q tests/test_g14_1_*.py

## 04. Drive mounting

In [ ]:
if not os.path.isdir(CONFIG['drive_root']):
    from google.colab import drive
    drive.mount('/content/drive')
print('drive root exists:', os.path.isdir(CONFIG['drive_root']))

## 05. Artifact discovery (R00)
Local first, then the Drive mirror used in place (nothing is regenerated because it is missing locally).

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
from gems_bench.g14_1.pipeline import G141Pipeline
pipe = G141Pipeline(CONFIG, g14_overrides=G14_OVERRIDES)
print('mode', pipe.cfg.mode, '| profile', pipe.cfg.profile, '| cfg hash', pipe.cfg.semantic_hash(), '| code', pipe.code_rev[:10])
display(pipe.plan())
pipe.run(stages=('R00',))
display(pd.read_csv(pipe.local / 'inventory' / 'preflight.csv'))
display(pipe.inv.groupby(['artifact', 'location']).size().rename('files').reset_index())

## 06. Geometry and submission parity (R01, P0)
Grid of every raster vs the template, metric kernel in pixels derived from the affine transform, hard-coded resolution assumptions, exported file vs rendered map (shift, flips, binarisation, footprint).

In [ ]:
pipe.run(stages=('R01',))
g = pipe.local / 'geometry_audit'
display(pd.read_csv(g / 'grid_table.csv'))
display(pd.read_csv(g / 'hard_coded_assumptions.csv'))
print(json.loads((g / 'kernel_footprint.json').read_text()))
if (g / 'submission_parity.csv').exists():
    display(pd.read_csv(g / 'submission_parity.csv').T)
print('ridge offset to known faults:', json.loads((g / 'ridge_offset_to_known.json').read_text()))

## 07. Existing G14 map inventory
Experiment-hash check of every training manifest and the feature-provenance table (which channels each checkpoint consumed, target-derived channels flagged).

In [ ]:
inv = pipe.inv
display(inv[inv.artifact.str.startswith('g14')][['artifact', 'location', 'rel', 'shape', 'grid_ok', 'bytes']])
display(pd.read_csv(pipe.local / 'inventory' / 'experiment_hash_check.csv'))
fp = pipe.local / 'inventory' / 'feature_provenance.csv'
if fp.exists():
    f = pd.read_csv(fp)
    display(f.groupby(['unit', 'source']).size().unstack(fill_value=0))

## 08. Raw probability diagnostics (R02)

In [ ]:
pipe.run(stages=('R02',))
d = pipe.local / 'probability_diagnostics'
if (d / 'full_summary.json').exists():
    s = json.loads((d / 'full_summary.json').read_text())
    display(pd.Series(s).to_frame('full map'))
    display(pd.read_csv(d / 'full_distance_curve.csv'))
    display(pd.read_csv(d / 'full_components.csv'))
if (d / 'heldout_ranking.csv').exists():
    display(pd.read_csv(d / 'heldout_ranking.csv'))

## 09. Known-context leakage audit (R02 static + R06 counterfactuals)
Static: share of training positives that are context pixels; region DTI of a predictor that only copies the context; AUC of distance-to-context on training vs held-out labels. Counterfactuals (mode COUNTERFACTUAL): the existing checkpoint with context disabled / neutral / shifted / subset / inner faults removed. A drop under perturbation is evidence of dependence, not proof of leakage on its own.

In [ ]:
L = pipe.local / 'leakage_audit'
for n in ('target_context_overlap', 'copy_baseline', 'separability'):
    if (L / f'{n}.csv').exists():
        display(Markdown(f'**{n}**')); display(pd.read_csv(L / f'{n}.csv'))
if pipe.cfg.stage_enabled('checkpoint_counterfactuals'):
    pipe.run(stages=('R06',))
    cf = pd.read_csv(L / 'counterfactuals.csv')
    display(cf.pivot_table(index=['unit', 'region', 'scoring'], columns='intervention', values='dti'))
else:
    print('counterfactuals need mode COUNTERFACTUAL and stages.checkpoint_counterfactuals=True')

## 10. Inner-validation methodology audit
The G14 'inner surrogate' is a mean of per-batch ratios on positive-centred patches. The region DTI sums official components over the whole inner region first. Both are shown side by side when R06 ran.

In [ ]:
r06 = pipe.local / 'reports' / 'R06.json'
if r06.exists():
    j = json.loads(r06.read_text())
    display(pd.DataFrame([j.get('inner', {}), j.get('counterfactual', {})], index=['inner', 'counterfactual']))
else:
    print('run section 09 in COUNTERFACTUAL mode first')

## 11. DTI component decomposition (R03 then R04)
Per held-out unit at its cross-fitted operating point: weighted TP / FP / FN, recall, precision diagnostic, area, components, skeleton length, localisation errors; failure classes F01-F10 with their evidence (classes without evidence are reported as not assessed).

In [ ]:
pipe.run(stages=('R03', 'R04'))
m = pipe.local / 'metric_decomposition'
if (m / 'g14_decomposition.csv').exists():
    display(pd.read_csv(m / 'g14_decomposition.csv').T)
    fc = pd.read_csv(m / 'failure_classes.csv')
    display(fc.pivot_table(index=['task', 'zone'], columns='class', values='status', aggfunc='first'))

## 12. Representation and area sweep
Requested vs actual (scored) area, dilation factor, masking order, soft calibrations; every choice cross-fitted (chosen on the other zones).

In [ ]:
r = pipe.local / 'representation_sweep'
if (r / 'g14_best_per_representation.csv').exists():
    display(pd.read_csv(r / 'g14_best_per_representation.csv').head(30))
    t = pd.read_csv(r / 'g14_sweep.csv')
    display(t[(t.rep == 'crest_w1') & (t.order == 'pixel_exact')][['task', 'zone', 'target_frac', 'final_frac', 'dilation_factor', 'dti']].head(40))
print(json.loads((pipe.local / 'reports' / 'R03.json').read_text()).get('tasks') if (pipe.local / 'reports' / 'R03.json').exists() else '')
for p in pipe.figures():
    if any(k in p.name for k in ('fig05', 'fig06', 'fig07')):
        display(Image(str(p)))

## 13. Geological failure atlas (R05b)

In [ ]:
if pipe.cfg.stage_enabled('failure_atlas'):
    pipe.run(stages=('R05b',))
    print('atlas index:', pipe.local / 'failure_atlas' / 'index.html')
    display(pd.read_csv(pipe.local / 'failure_atlas' / 'atlas_index.csv').head(30))

## 14. Champion comparison (R05)
Scored comparisons only on identical units (g13 backend); full maps compared label-free.

In [ ]:
pipe.run(stages=('R05',))
c = pipe.local / 'champion_comparison'
for n in ('label_free_full_maps', 'champion_best_per_rep'):
    if (c / f'{n}.csv').exists():
        display(Markdown(f'**{n}**')); display(pd.read_csv(c / f'{n}.csv').head(30))

## 15. Complementarity and residual recovery (R12)
Champion first; G14 weights 0-0.5; weight for each zone chosen on the other zones; weight 0 (champion alone) always a candidate.

In [ ]:
pipe.run(stages=('R12',))
k = pipe.local / 'complementarity'
for n in ('pixel_classes', 'blend_crossfit_geo', 'blend_crossfit_disc_official', 'regime_conditional_blend'):
    if (k / f'{n}.csv').exists():
        display(Markdown(f'**{n}**')); display(pd.read_csv(k / f'{n}.csv'))
print(json.loads((pipe.local / 'reports' / 'R12.json').read_text()) if (pipe.local / 'reports' / 'R12.json').exists() else '')

## 16. Corrected discovery splits (R07)

In [ ]:
if not pipe.cfg.stage_enabled('discovery_splits'):
    pipe.cfg = pipe.cfg.updated({'stages': {'discovery_splits': True}})
pipe.run(stages=('R07',))
s = pipe.local / 'discovery_splits'
display(pd.read_csv(s / 'protocol_check.csv'))
display(pd.read_csv(s / 'split_audit.csv')[['unit', 'train_px', 'inner_val_px', 'heldout_px', 'train_pos_in_context', 'status']])

## 17. Corrected target construction
Separated train / inner targets with a label-free boundary embargo; conditional context / target draws (Y_target = Y_visible minus Y_context).

In [ ]:
display(pd.read_csv(s / 'separated_target_checks.csv'))
display(pd.read_csv(s / 'conditional_draws.csv'))

## 18. Loss comparison (R11, mode TRAIN)
Selectable objectives, negative policies, augmentation policies and feature families, each changing one factor against M1.

In [ ]:
from gems_bench.g14_1.experiments import matrix
display(matrix())
if pipe.cfg.allow_training and pipe.cfg.stage_enabled('train_models'):
    pipe.run(stages=('R11',))

## 19. Known-context-only baseline (R08, D1) and the original control (M0)

In [ ]:
if pipe.cfg.allow_training and pipe.cfg.stage_enabled('train_models'):
    pipe.run(stages=('R08',))
else:
    print('mode TRAIN + stages.train_models=True required')

## 20. Geophysics-only baseline (R09, M1 / D2)

In [ ]:
if pipe.cfg.allow_training and pipe.cfg.stage_enabled('train_models'):
    pipe.run(stages=('R09',))

## 21. Conditional discovery training (R10, M2 + M3)

In [ ]:
if pipe.cfg.allow_training and pipe.cfg.stage_enabled('train_models'):
    pipe.run(stages=('R10',))
sc = pipe.local / 'corrected_models' / 'scorecard.csv'
if sc.exists():
    display(pd.read_csv(sc))

## 22. Local-global architecture experiments (R13-R15)
Only meaningful against the corrected local CNN with identical features, supervision, folds, masks and optimisation (`experiments.REGISTRY` changes one factor).

In [ ]:
if pipe.cfg.allow_training and pipe.cfg.stage_enabled('train_models'):
    pipe.run(stages=('R13',))
if pipe.cfg.allow_training and pipe.cfg.stage_enabled('transformers'):
    pipe.run(stages=('R14', 'R15'))

## 23. Discovery calibration (R16 and the selected operating points)

In [ ]:
if pipe.cfg.allow_training:
    pipe.run(stages=('R16',))
sel = pipe.local / 'calibration' / 'selected_operating_points.json'
if sel.exists():
    j = json.loads(sel.read_text())
    display(pd.DataFrame([{'model': m, 'task': t, **v} for m, d in j.items() for t, v in d.items()]))

## 24. Final candidate ranking
Discovery-calibrated candidates first; geographic-only calibrations are PROVISIONAL.

In [ ]:
if sel.exists():
    rk = pd.DataFrame([{'model': m, 'task': t, **v} for m, d in j.items() for t, v in d.items()])
    rk['status'] = rk.task.map(lambda t: 'DISCOVERY_CALIBRATED' if t.startswith('disc') else 'PROVISIONAL_GEO')
    display(rk.sort_values(['status', 'crossfit_dti'], ascending=[True, False]))

## 25. Official submission preparation (mode SUBMIT)
One cell. Reuses the full map and the best operating point of the SAME model; writes the GeoTIFF, validation, parity check and provenance; syncs to Drive. Upload by hand; do not tune on the leaderboard.

In [ ]:
SUBMIT_MODEL = 'g14'        # 'g14' | 'g14_1:<spec>' | 'g13:<model>' | 'p6_w2' | a .npy / .tif path
SUBMIT_KW = {}              # e.g. {'rep': 'sparse_soft', 'frac': 0.01}  -> labelled MANUAL
if pipe.cfg.mode == 'SUBMIT':
    rec = pipe.submission(SUBMIT_MODEL, **SUBMIT_KW)
    print(rec['status'], '| valid:', rec['validation']['valid'], rec['validation']['checks_failed'])
    print('file:', rec['path']); print('drive:', rec.get('drive_path'))
else:
    print("set CONFIG['mode'] = 'SUBMIT' and rerun cells 01 and 05")

## 26. Google Drive synchronization

In [ ]:
print('files copied to Drive:', pipe.sync())
display(pipe.ledger.table())

## 27. Final diagnostic report
Hypotheses H_A-H_G with supporting / contradictory evidence and outcomes A-G. Status NOT_EVALUATED means the evidence has not been produced yet; nothing is inferred from missing evidence.

In [ ]:
p = pipe.report()
display(pd.read_csv(p.with_suffix('.csv'))[['hypothesis', 'status', 'supporting_evidence', 'contradictory_evidence', 'recommended_action']])
print('report:', p)
display(pd.read_csv(pipe.local / 'reports' / 'figures_index.csv'))